# Mars lava-tube / skylight candidate detection (JP2)

This notebook scans a **single JP2** image and branches automatically between **IMAGERY** and **DEM/DTM** products.
It generates extensive debug outputs, candidate CSV/GeoJSON, and a mask raster in Drive.

## 1) Mount Google Drive + paths

In [ ]:
from google.colab import drive
import os
import time
from pathlib import Path

# Mount Drive
if not os.path.exists('/content/drive'):
    os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive')

# Fixed project paths
proj = "/content/drive/MyDrive/MARS_LAVATUBES"
hirise_dir = f"{proj}/HIRISE_DTMS"

# Run naming
run_name = time.strftime("%Y%m%d_%H%M%S")
out_dir = f"{proj}/outputs/jp2_scans/{run_name}"
Path(out_dir).mkdir(parents=True, exist_ok=True)

# Default input path (user editable)
input_jp2_path = f"{hirise_dir}/YOUR_INPUT.jp2"

print("Project:", proj)
print("HiRISE dir:", hirise_dir)
print("Output dir:", out_dir)
print("Input JP2:", input_jp2_path)

# Validate input
input_path = Path(input_jp2_path)
if not input_path.exists():
    raise FileNotFoundError(
        f"Input JP2 not found: {input_jp2_path}
"
        "Please update input_jp2_path to a valid JP2 file in Drive."
    )

print("Input size (MB):", round(input_path.stat().st_size / (1024 * 1024), 2))

## 2) Parameter block (edit here)

In [ ]:
# ==========================
# Configuration Parameters
# ==========================

# Input / output
input_jp2_path = input_jp2_path  # editable path
run_name = run_name              # set above
out_dir = out_dir                # set above
band_index = 1                   # choose which band to read (1-based)

# Preprocessing
seed = 42
blur_sigma = 0.8           # set to 0 to disable
redownsample_if_huge = True
max_pixels = 25_000_000     # auto-downsample if image too large

downsample_factor = 1       # manual downsample factor (>=1)

# Branch override: "AUTO" | "IMAGERY" | "DEM"
branch_override = "AUTO"

# Imagery thresholds (dark-pit detection)
min_area = 20
max_area = 5000
min_contrast = 0.08
min_circularity = 0.35
max_eccentricity = 0.85
log_sigma_min = 1.0
log_sigma_max = 6.0
num_sigma = 8
ring_inner = 1.5
ring_outer = 3.0
use_blackhat = True
blackhat_size = 15

# DEM thresholds (terrain geometry)
neighborhood_size = 21
min_depth = 0.8
slope_max = 35.0
curvature_threshold = -0.05

print("Parameters loaded.")


## 3) Imports + dependency helpers

In [ ]:
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import time


def _run(cmd):
    print("
$", cmd)
    return subprocess.check_call(cmd, shell=True)


def ensure_base_deps():
    try:
        import rasterio  # noqa: F401
        return
    except Exception:
        print("rasterio missing; installing base deps...")
        _run("pip -q install rasterio opencv-python scikit-image scipy matplotlib glymur pillow")


def install_gdal_and_retry():
    print("JP2 driver missing or rasterio failed. Installing GDAL + deps...")
    _run("apt-get -qq update")
    _run("apt-get -qq install -y gdal-bin libgdal-dev")
    _run("pip -q install rasterio opencv-python scikit-image scipy matplotlib glymur pillow")


ensure_base_deps()


## 4) Robust JP2 reading with fallbacks

In [ ]:
import numpy as np


def read_jp2(path, band_index=1, downsample_factor=1):
    # Returns: data (2D float32), transform, crs, tags, nodata, scale_factor
    try:
        import rasterio
        from rasterio.enums import Resampling
    except Exception as exc:
        raise RuntimeError("rasterio import failed even after install") from exc

    try:
        with rasterio.open(path) as src:
            tags = src.tags()
            nodata = src.nodata
            transform = src.transform
            crs = src.crs
            count = src.count
            if band_index < 1 or band_index > count:
                print(f"Band index {band_index} out of range (1..{count}). Using band 1.")
                band_index = 1

            scale_factor = downsample_factor
            if downsample_factor > 1:
                out_shape = (
                    int(src.height / downsample_factor),
                    int(src.width / downsample_factor),
                )
                data = src.read(
                    band_index,
                    out_shape=out_shape,
                    resampling=Resampling.bilinear,
                )
                transform = src.transform * src.transform.scale(
                    (src.width / out_shape[1]),
                    (src.height / out_shape[0])
                )
            else:
                data = src.read(band_index)

        return data.astype(np.float32), transform, crs, tags, nodata, scale_factor

    except Exception as exc:
        print("Rasterio failed:", exc)
        install_gdal_and_retry()
        try:
            with rasterio.open(path) as src:
                tags = src.tags()
                nodata = src.nodata
                transform = src.transform
                crs = src.crs
                count = src.count
                if band_index < 1 or band_index > count:
                    band_index = 1
                if downsample_factor > 1:
                    out_shape = (
                        int(src.height / downsample_factor),
                        int(src.width / downsample_factor),
                    )
                    data = src.read(
                        band_index,
                        out_shape=out_shape,
                        resampling=rasterio.enums.Resampling.bilinear,
                    )
                    transform = src.transform * src.transform.scale(
                        (src.width / out_shape[1]),
                        (src.height / out_shape[0])
                    )
                else:
                    data = src.read(band_index)
            return data.astype(np.float32), transform, crs, tags, nodata, downsample_factor
        except Exception as exc2:
            print("Rasterio retry failed:", exc2)
            print("Trying glymur fallback...")
            import glymur
            jp2 = glymur.Jp2k(path)
            data = jp2[:]
            if data.ndim == 3:
                data = data[:, :, band_index - 1]
            return data.astype(np.float32), None, None, {}, None, downsample_factor


# Auto-downsample if huge
input_path = Path(input_jp2_path)
forced_downsample = downsample_factor
if redownsample_if_huge:
    try:
        import rasterio
        with rasterio.open(input_path) as src:
            total_pixels = src.width * src.height
        if total_pixels > max_pixels and downsample_factor == 1:
            forced_downsample = int(np.ceil(np.sqrt(total_pixels / max_pixels)))
            print(f"Auto downsample factor set to {forced_downsample} for large raster.")
    except Exception:
        pass

# Read data
raw, transform, crs, tags, nodata, scale_factor = read_jp2(
    str(input_path), band_index=band_index, downsample_factor=forced_downsample
)

print("Loaded JP2:", raw.shape, "dtype:", raw.dtype)
print("CRS:", crs)
print("Transform:", transform)
print("Tags:", tags)
print("Band index:", band_index)
print("Scale factor:", scale_factor)


## 5) Preprocessing utilities

In [ ]:
from scipy import ndimage
import numpy as np

np.random.seed(seed)


def handle_nodata(data, nodata):
    data = data.astype(np.float32)
    if nodata is not None:
        data = np.where(data == nodata, np.nan, data)
    data = np.where(np.isfinite(data), data, np.nan)
    return data


def normalize_percentile(data, p2=2, p98=98):
    valid = data[np.isfinite(data)]
    if valid.size == 0:
        return data
    lo, hi = np.percentile(valid, (p2, p98))
    if hi - lo < 1e-6:
        return np.clip(data, lo, hi)
    out = (data - lo) / (hi - lo)
    return np.clip(out, 0, 1)


def apply_blur(data, sigma):
    if sigma and sigma > 0:
        return ndimage.gaussian_filter(data, sigma=sigma)
    return data


def quicklook_image(data, is_dem=False):
    if is_dem:
        return normalize_percentile(data)
    return normalize_percentile(data)

## 6) Auto-detect product type

In [ ]:
import numpy as np


def detect_product_type(data, tags, transform, override="AUTO"):
    evidence = []

    if override in ["IMAGERY", "DEM"]:
        return override, [f"Manual override: {override}"]

    tag_text = " ".join([f"{k}:{v}" for k, v in tags.items()]).lower()
    if any(k in tag_text for k in ["dtm", "dem", "elevation", "height"]):
        evidence.append("Tags indicate DEM/DTM")
        return "DEM", evidence

    valid = data[np.isfinite(data)]
    if valid.size == 0:
        evidence.append("No valid data; defaulting to IMAGERY")
        return "IMAGERY", evidence

    vmin, vmax = np.nanmin(valid), np.nanmax(valid)
    vstd = np.nanstd(valid)
    p2, p98 = np.nanpercentile(valid, [2, 98])
    evidence.append(f"min={vmin:.2f} max={vmax:.2f} std={vstd:.2f} p2={p2:.2f} p98={p98:.2f}")

    grad = np.gradient(valid.astype(np.float32))
    grad_mag = np.sqrt(grad[0]**2 + grad[1]**2)
    grad_stats = (float(np.nanmedian(grad_mag)), float(np.nanpercentile(grad_mag, 95)))
    evidence.append(f"grad_median={grad_stats[0]:.4f}, grad_p95={grad_stats[1]:.4f}")

    range_val = vmax - vmin
    if range_val > 100 and vstd > 5:
        evidence.append("Range/std suggest DEM/DTM")
        return "DEM", evidence

    evidence.append("Defaulting to IMAGERY (uncertain)")
    return "IMAGERY", evidence


product_type, evidence = detect_product_type(raw, tags, transform, override=branch_override)
print("Detected product type:", product_type)
print("Evidence:")
for e in evidence:
    print(" -", e)

## 7) Preprocess input

In [ ]:
start_time = time.time()

raw = handle_nodata(raw, nodata)

if product_type == "DEM":
    dem_data = raw.copy()
    dem_vis = normalize_percentile(dem_data)
else:
    img_data = raw.copy()
    img_vis = normalize_percentile(img_data)

if product_type == "DEM":
    dem_data = apply_blur(dem_data, blur_sigma)
else:
    img_data = apply_blur(img_data, blur_sigma)

plt.figure(figsize=(6, 6))
plt.imshow(dem_vis if product_type == "DEM" else img_vis, cmap='gray')
plt.title("Input quicklook (normalized)")
plt.axis('off')
plt.tight_layout()
plt.savefig(f"{out_dir}/quicklook.png", dpi=200)
plt.close()

plt.figure(figsize=(6, 4))
vals = raw[np.isfinite(raw)].ravel()
plt.hist(vals, bins=200, color='steelblue', alpha=0.8)
plt.title("Value histogram")
plt.xlabel("Value")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(f"{out_dir}/histogram.png", dpi=200)
plt.close()

print("Quicklook + histogram saved.")

## 8) DEM/DTM branch: terrain-geometry detection

In [ ]:
from scipy import ndimage
from skimage import measure, morphology


def dem_pipeline(dem, transform):
    print("Running DEM pipeline...")
    print(
        f"Params: neighborhood_size={neighborhood_size}, min_depth={min_depth}, "
        f"slope_max={slope_max}, curvature_threshold={curvature_threshold}, min_area={min_area}"
    )

    if transform is not None:
        pixel_x = abs(transform.a)
        pixel_y = abs(transform.e)
    else:
        pixel_x = pixel_y = 1.0

    gy, gx = np.gradient(dem, pixel_y, pixel_x)
    slope = np.degrees(np.arctan(np.sqrt(gx**2 + gy**2)))

    curvature = ndimage.laplace(dem)

    size = neighborhood_size
    local_mean = ndimage.uniform_filter(dem, size=size, mode='nearest')
    depth = local_mean - dem

    local_min = (dem == ndimage.minimum_filter(dem, size=size))
    print("Local minima count:", int(np.sum(local_min)))

    mask_depth = depth >= min_depth
    mask_slope = slope <= slope_max
    mask_curve = curvature <= curvature_threshold

    print("Depth filter count:", int(np.sum(local_min & mask_depth)))
    print("Slope filter count:", int(np.sum(local_min & mask_depth & mask_slope)))
    print("Curvature filter count:", int(np.sum(local_min & mask_depth & mask_slope & mask_curve)))

    pit_mask = (
        local_min &
        mask_depth &
        mask_slope &
        mask_curve
    )

    pit_mask = morphology.remove_small_objects(pit_mask, min_size=min_area)
    print("After min_area filter:", int(np.sum(pit_mask)))

    labels = measure.label(pit_mask)
    props = measure.regionprops(labels, intensity_image=depth)

    candidates = []
    for p in props:
        cy, cx = p.centroid
        area = p.area
        mean_depth = float(np.nanmean(depth[labels == p.label]))
        score = mean_depth * np.sqrt(area)
        candidates.append({
            "id": p.label,
            "x_px": float(cx),
            "y_px": float(cy),
            "score": float(score),
            "area": float(area),
            "depth": float(mean_depth),
            "slope": float(np.nanmean(slope[labels == p.label])),
            "curvature": float(np.nanmean(curvature[labels == p.label]))
        })

    plt.figure(figsize=(6, 6))
    plt.imshow(slope, cmap='magma')
    plt.title("Slope (deg)")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/slope.png", dpi=200)
    plt.close()

    plt.figure(figsize=(6, 6))
    plt.imshow(curvature, cmap='coolwarm')
    plt.title("Curvature (Laplacian)")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/curvature.png", dpi=200)
    plt.close()

    plt.figure(figsize=(6, 6))
    plt.imshow(depth, cmap='inferno')
    plt.title("Depth (local mean - elevation)")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/depth.png", dpi=200)
    plt.close()

    return pit_mask, candidates, slope, curvature, depth


## 9) IMAGERY branch: photometric pit/skylight detection

In [ ]:
from skimage import feature, morphology, measure
from scipy import ndimage


def compute_contrast(image, y, x, radius, ring_inner, ring_outer):
    yy, xx = np.ogrid[:image.shape[0], :image.shape[1]]
    dist = np.sqrt((yy - y) ** 2 + (xx - x) ** 2)
    inner = dist <= radius
    ring = (dist >= radius * ring_inner) & (dist <= radius * ring_outer)
    if np.any(inner) and np.any(ring):
        return float(np.nanmean(image[ring]) - np.nanmean(image[inner]))
    return 0.0


def imagery_pipeline(image):
    print("Running IMAGERY pipeline...")
    print(
        "Params: min_area={}, max_area={}, min_contrast={}, min_circularity={}, "
        "max_eccentricity={}, log_sigma_min={}, log_sigma_max={}, num_sigma={}, "
        "ring_inner={}, ring_outer={}, use_blackhat={}".format(
            min_area, max_area, min_contrast, min_circularity, max_eccentricity,
            log_sigma_min, log_sigma_max, num_sigma, ring_inner, ring_outer, use_blackhat
        )
    )
    img = normalize_percentile(image)

    if use_blackhat:
        selem = morphology.disk(blackhat_size)
        bh = morphology.black_tophat(img, selem)
        img_proc = normalize_percentile(bh)
    else:
        bh = None
        img_proc = img

    inv = 1.0 - img_proc
    blobs = feature.blob_log(
        inv,
        min_sigma=log_sigma_min,
        max_sigma=log_sigma_max,
        num_sigma=num_sigma,
        threshold=0.03
    )

    print("Blobs detected (LoG):", len(blobs))

    sigmas = np.linspace(log_sigma_min, log_sigma_max, num_sigma)
    response = np.zeros_like(img_proc, dtype=np.float32)
    for s in sigmas:
        resp = -ndimage.gaussian_laplace(img_proc, s)
        response = np.maximum(response, resp)

    # Contrast map (local difference)
    small = ndimage.uniform_filter(img_proc, size=max(3, int(ring_inner * 4)))
    large = ndimage.uniform_filter(img_proc, size=max(5, int(ring_outer * 8)))
    contrast_map = large - small

    candidates = []
    mask = np.zeros_like(img_proc, dtype=bool)

    area_pass = 0
    contrast_pass = 0
    shape_pass = 0

    for i, (y, x, sigma) in enumerate(blobs, start=1):
        radius = sigma * np.sqrt(2)
        y = int(round(y))
        x = int(round(x))

        if y <= 0 or x <= 0 or y >= img_proc.shape[0] - 1 or x >= img_proc.shape[1] - 1:
            continue

        area = np.pi * (radius ** 2)
        if area < min_area or area > max_area:
            continue
        area_pass += 1

        contrast = compute_contrast(img_proc, y, x, radius, ring_inner, ring_outer)
        if contrast < min_contrast:
            continue
        contrast_pass += 1

        rr, cc = np.ogrid[:img_proc.shape[0], :img_proc.shape[1]]
        circle = (rr - y) ** 2 + (cc - x) ** 2 <= radius ** 2
        label = measure.label(circle)
        props = measure.regionprops(label)
        if not props:
            continue
        p = props[0]

        circularity = 4 * np.pi * p.area / (p.perimeter ** 2 + 1e-6)
        eccentricity = p.eccentricity

        if circularity < min_circularity or eccentricity > max_eccentricity:
            continue
        shape_pass += 1

        score = float(contrast * circularity * np.sqrt(area))

        mask |= circle
        candidates.append({
            "id": len(candidates) + 1,
            "x_px": float(x),
            "y_px": float(y),
            "score": float(score),
            "area": float(area),
            "circularity": float(circularity),
            "eccentricity": float(eccentricity),
            "contrast": float(contrast)
        })

    print("After area filter:", area_pass)
    print("After contrast filter:", contrast_pass)
    print("After shape filter:", shape_pass)

    plt.figure(figsize=(6, 6))
    plt.imshow(response, cmap='viridis')
    plt.title("LoG response")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/log_response.png", dpi=200)
    plt.close()

    plt.figure(figsize=(6, 6))
    plt.imshow(contrast_map, cmap='magma')
    plt.title("Contrast map")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(f"{out_dir}/contrast_map.png", dpi=200)
    plt.close()

    if bh is not None:
        plt.figure(figsize=(6, 6))
        plt.imshow(bh, cmap='gray')
        plt.title("Black-hat output")
        plt.colorbar()
        plt.tight_layout()
        plt.savefig(f"{out_dir}/blackhat.png", dpi=200)
        plt.close()

    return mask, candidates, response


## 10) Run detection branch + overlay

In [ ]:
if product_type == "DEM":
    pit_mask, candidates, slope, curvature, depth = dem_pipeline(dem_data, transform)
    branch = "DEM"
else:
    pit_mask, candidates, response = imagery_pipeline(img_data)
    branch = "IMAGERY"

print("Total candidates:", len(candidates))

plt.figure(figsize=(7, 7))
base = dem_vis if product_type == "DEM" else img_vis
plt.imshow(base, cmap='gray')
for c in candidates:
    plt.plot(c["x_px"], c["y_px"], 'r+', markersize=8)
    plt.text(c["x_px"] + 2, c["y_px"] + 2, str(c["id"]), color='yellow', fontsize=8)
plt.title(f"Candidates overlay ({branch})")
plt.axis('off')
plt.tight_layout()
plt.savefig(f"{out_dir}/candidates_overlay.png", dpi=200)
plt.close()

## 11) Export outputs (CSV, GeoJSON, mask raster)

In [ ]:
import csv
import json

csv_path = f"{out_dir}/candidates.csv"

base_fields = [
    "id", "x_px", "y_px", "x_px_ds", "y_px_ds", "score", "area",
    "circularity", "eccentricity", "contrast", "branch"
]
extra_fields = []
if branch == "DEM":
    extra_fields = ["depth", "slope", "curvature"]

fields = base_fields + extra_fields

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for c in candidates:
        x_ds = c.get("x_px", 0.0)
        y_ds = c.get("y_px", 0.0)
        row = {k: c.get(k, "") for k in fields}
        row["x_px_ds"] = x_ds
        row["y_px_ds"] = y_ds
        row["x_px"] = float(x_ds) * scale_factor
        row["y_px"] = float(y_ds) * scale_factor
        row["branch"] = branch
        row.setdefault("circularity", "")
        row.setdefault("eccentricity", "")
        row.setdefault("contrast", "")
        writer.writerow(row)

print("Saved:", csv_path)

geojson_path = f"{out_dir}/candidates.geojson"
if transform is not None and crs is not None:
    features = []
    for c in candidates:
        x, y = c["x_px"], c["y_px"]
        gx, gy = transform * (x, y)
        props = {k: v for k, v in c.items() if k not in ["x_px", "y_px"]}
        props["branch"] = branch
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [gx, gy]},
            "properties": props
        })

    geojson = {
        "type": "FeatureCollection",
        "features": features,
        "crs": {"type": "name", "properties": {"name": str(crs)}}
    }

    with open(geojson_path, "w") as f:
        json.dump(geojson, f, indent=2)
    print("Saved:", geojson_path)
else:
    print("No georeference found; GeoJSON not written.")

mask_path = f"{out_dir}/candidates_mask.tif"
try:
    import rasterio
    from rasterio.transform import from_origin
    if transform is None:
        transform = from_origin(0, 0, 1, 1)

    profile = {
        "driver": "GTiff",
        "height": pit_mask.shape[0],
        "width": pit_mask.shape[1],
        "count": 1,
        "dtype": "uint8",
        "transform": transform,
        "crs": crs
    }

    with rasterio.open(mask_path, "w", **profile) as dst:
        dst.write(pit_mask.astype(np.uint8), 1)
    print("Saved:", mask_path)
except Exception as exc:
    print("Rasterio export failed; saving PNG mask. Error:", exc)
    from PIL import Image
    Image.fromarray((pit_mask.astype(np.uint8) * 255)).save(f"{out_dir}/candidates_mask.png")


## 12) Summary

In [ ]:
end_time = time.time()

print("=== Summary ===")
print("Branch:", branch)
print("Image shape:", raw.shape)
print("Candidates:", len(candidates))
print("Runtime (s):", round(end_time - start_time, 2))
print("Output dir:", out_dir)

if len(candidates) == 0:
    print("No candidates detected. Consider relaxing thresholds or checking branch selection.")

## 13) Output inventory


In [ ]:
from pathlib import Path

print("Listing outputs in: ", out_dir)
out_path = Path(out_dir)
if out_path.exists():
    pngs = sorted(out_path.glob('*.png'))
    tifs = sorted(out_path.glob('*.tif'))
    others = sorted(out_path.glob('*.csv')) + sorted(out_path.glob('*.geojson'))
    print("PNGs:", [p.name for p in pngs])
    print("TIFFs:", [p.name for p in tifs])
    print("Other files:", [p.name for p in others])
else:
    print("Output directory not found.")
